# Quickstart

## Your First Kafi Streams Topology

As in Kafka Streams, in Kafi Streams, a processing pipeline is called *topology*.

A Kafi Streams topology is a directed acyclic graph of operators starting with arbitrary many *sources* and ending with arbitrary many *sinks* (typically corresponding to Kafka topics).

Here is a concrete example, displayed in the traditional Kafka Streams-like way (see https://zz85.github.io/kafka-streams-viz/):

```mermaid
graph TD
25aa7736-238a-40f0-8054-d9c6f986ee3b[source_clicks] --> a463ea6a-2d84-4e5d-bbf6-6d7cd7a22eb5[map_op]
fcb627f1-8b99-408c-941a-130bfbf828c4[map_op] --> 536f3bac-98f3-4b53-bb64-e3874777f1ce[join_equi_op]
536f3bac-98f3-4b53-bb64-e3874777f1ce[join_equi_op] --> 56b14bac-535b-4bfa-81ae-6347726eb5f9[sink_joined]
14a0f926-3b42-4f6a-ad3a-0f0fc40138b7[filter_op] --> 536f3bac-98f3-4b53-bb64-e3874777f1ce[join_equi_op]
a463ea6a-2d84-4e5d-bbf6-6d7cd7a22eb5[map_op] --> 14a0f926-3b42-4f6a-ad3a-0f0fc40138b7[filter_op]
2c812b6a-4b42-4293-a143-b36e797d0a84[source_customers] --> fcb627f1-8b99-408c-941a-130bfbf828c4[map_op]
```

There are two sources (`source_clicks` and `source_customers`) at the top. Each source goes through a `map` operaotor. The clicks, in addition, go through a `filter` operator. Then, both sides are joined by the `join_equi` operator and end up in the sink (`sink_joined`).



## How Does the Data Look Like?

We assume that `clicks` is a source of Kafka messages like this:

In [ ]:
click_m_dict = {
    "key": None,
    "value": {"customer_id": "4711",
              "view_time": 200,
              "ts": 1609457200000},
    "partition": 2,
    "offset": 23,
    "timestamp": 1609457201000,
    "headers": None
}

...and `customers` is a source of Kafka messages like this:

In [ ]:
customer_m_dict = {
    "key": "4711",
    "value": {"id": "4711",
              "name": "Sallyann Jupp"},
    "partition": 0,
    "offset": 67,
    "timestamp": 1609457001000,
    "headers": None
}

The aim of the topology is to join `clicks` with `customers` to enrich the `customer_id` in the `clicks` with the `name` of the customer from the `customers`. An example output message looks like this (only the `value` field of the Kafka message is relevant here):

```python
{"value": {"customer_id": "4711",
           "view_time": 200,
           "name": "Sallyann Jupp"}
}
```

## Setting Up the Topology

At first, let's import `Streams` and set up the logging:
```python
from kafi.streams.streams import Streams
import logging
logging.basicConfig(level=logging.INFO)

```

Secondly, we need a Kafka. If you have a local Kafka running, you can use that:
```python
from kafi.kafka.cluster.cluster import Cluster
c = Cluster({"kafka": {"bootstrap.servers": "localhost:9092"}})
```

Or, if you don't, just go for Kafi's emulated Kafka on your local disk:
```python
from kafi.fs.local.local import Local
c = Local({"local": {"root.dir": "/tmp"}})
```

Before we build our topology, we specify the names of the source and sink topics:
```python
click_source_str = "clicks"
customer_source_str = "customers"
sink_str = "joined"
```

### Clicks

The topology consists of three parts. The first part of the topology is about the clicks:
```python
click_tn = (
    Streams.source(c, click_source_str) # 1.
    .map(lambda r: {"customer_id": r["value"]["customer_id"], "view_time": r["value"]["view_time"]}) # 2.
    .filter(lambda r: r["view_time"] < 100) # 3.
)
```

What happens here?
1. We define the source. Here, `c` stands for the Kafka cluster, `click_source_str` is the topic name. As you can see, contrary to Kafka Streams, all sources and sinks can be on any Kafka cluster.
2. We use the `map` operator to select two fields from the value of the incoming click events (`customer_id` and `view_time`).
3. We employ the `filter` operator to filter out those incoming click events where `view_time` is smaller than `100`.

As for the naming conventions used throughout:
* `_tn` is a suffix for instances of the Kafi Streams `TopologyNode` class.
* `r` stands for "record" (=typically a Python dictionary), `_r` is the corresponding suffix.

### Customers

The second part of the topology is about the customers:
```python
customer_tn = (
    Streams.source(c, customer_source_str)
    .map(lambda r: {"id": r["value"]["id"], "name": r["value"]["name"]})
)
```

What happens?
1. We define the source.
2. We select two fields from the value of the incoming customer messages (`id` and `name`).

### Join and sink

The third and last part of the topology joins the clicks and customers using an equi join (`join_equi`) and sinks the result.

```python
sink_tn = (
    click_tn
    .join_equi(
        customer_tn,
        lambda l_r: l_r["customer_id"],
        lambda r_r: r_r["id"],
        lambda l_r, r_r: {"value": {
            "customer_id": l_r["customer_id"],
            "view_time": l_r["view_time"],
            "name": r_r["name"]}})
    .sink(c, sink_str)
)
```

The arguments of the `join_equi()` operator are:
1. Right side of the join (here: `customer_tn`)
2. Left side join key (here: the `customer_id` field of the left side = the clicks)
3. Right side join key (here: the `id` field of the right side = the customers)
4. Projection function. Here: `customer_id` and `view_time` from the clicks, and `name` from the customers.

At the end of the topology is the sink definition (cluster `c` and topic name `sink_str`).

### Building

Before you can use a Kafi Streams topology, it needs to be "built". This basically creates a pydbsp "circuit" that eventually does the processing:
```python
built_tn = Streams.build(sink_tn)
```

That's it. Here is the full code:


In [1]:
import sys
sys.path.insert(1, "..")

import importlib
import kafi.streams.topologynode
import kafi.streams.streams
importlib.reload(kafi.streams.topologynode)
importlib.reload(kafi.streams.streams)

from kafi.streams.streams import Streams
import logging
logging.basicConfig(level=logging.INFO)

# Kafka: Option 1: Connect to real local Kafka instaed
from kafi.kafka.cluster.cluster import Cluster
c = Cluster({"kafka": {"bootstrap.servers": "localhost:9092"}})

# Kafka: Option 2: "Connect" to Kafi's Kafka emulation on local disk
# from kafi.fs.local.local import Local
# c = Local({"local": {"root.dir": "/tmp"}})

#

click_source_str = "clicks"
customer_source_str = "customers"
sink_str = "joined"
#
click_tn = (
    Streams.source(c, click_source_str)
    .map(lambda r: {"customer_id": r["value"]["customer_id"], "view_time": r["value"]["view_time"]})
    .filter(lambda r: r["view_time"] > 20)
)
#
customer_tn = (
    Streams.source(c, customer_source_str)
    .map(lambda r: {"id": r["value"]["id"], "name": r["value"]["name"]})
)
#
sink_tn = (
    click_tn
    .join_equi(
        customer_tn,
        lambda l_r: l_r["customer_id"],
        lambda r_r: r_r["id"],
        lambda l_r, r_r: {"value": {
            "customer_id": l_r["customer_id"],
            "view_time": l_r["view_time"],
            "name": r_r["name"]}})
    .sink(c, sink_str)
)
#
b_tn = Streams.build(sink_tn)


Kafi Streams offers two ways for visualizing topologies
* `topology()`: Bracketed visualization
* `mermaid()`: Mermaid-based visualization (can be pasted e.g. into Markdown files)

In [2]:
print("topology()")
print(b_tn.topology())
print("")
print("mermaid()")
print(b_tn.mermaid())


topology()
sink_joined(join_equi_op(filter_op(map_op(source_clicks)), map_op(source_customers)))

mermaid()
```mermaid
graph TD
7e64c84f-e93a-4dd6-a54c-528f9ef2424a[join_equi_op] --> 07c34c81-3d8f-43d7-bf81-4339da7aba5f[sink_joined]
6732a127-5d01-4824-b845-eb0d86d8ba8f[filter_op] --> 7e64c84f-e93a-4dd6-a54c-528f9ef2424a[join_equi_op]
44de9f78-6a74-40c5-abd4-631da2ecf2e5[source_clicks] --> b63fb38c-ad1a-48c0-9a28-a2edd404c5c3[map_op]
b63fb38c-ad1a-48c0-9a28-a2edd404c5c3[map_op] --> 6732a127-5d01-4824-b845-eb0d86d8ba8f[filter_op]
5aac8cc5-47fb-456c-8dcf-d7d90423d4e8[source_customers] --> afa7dad9-472b-4e4b-a444-edf4a603d0ad[map_op]
afa7dad9-472b-4e4b-a444-edf4a603d0ad[map_op] --> 7e64c84f-e93a-4dd6-a54c-528f9ef2424a[join_equi_op]
```


## Let's Generate Some Data

Next, you'd probably like to see the topology in action. So let's write data generators for the clicks and the customers.


In [2]:
import random, time

from faker import Faker

customers_int = 100

class ClickGenerator:
    def __init__(self):
        self.ts_int = int(time.time() * 1000)
        self.ts_step_int = 100
        self.customer_id_int = 0

    def generate(self):
        message_dict = {
            "key": None,
            "value": {"customer_id": random.randint(0, customers_int - 1),
                      "view_time": random.randint(10, 120),
                      "ts": self.ts_int},
        }
        #
        self.ts_int += self.ts_step_int
        #
        return message_dict

class CustomerGenerator:
    def __init__(self):
        self.customer_id_int = 0
        self.customer_id_int_name_str_dict = {}
        fake = Faker()
        for customer_id_int in range(customers_int):
            name_str = fake.name()
            self.customer_id_int_name_str_dict[customer_id_int] = name_str

    def generate(self):
        customer_id_int = random.randint(0, customers_int - 1)
        message_dict = {
            "key": str(customer_id_int),
            "value": {"id": customer_id_int,
                      "name": self.customer_id_int_name_str_dict[customer_id_int]}
        }
        #
        return message_dict
    
click_generator = ClickGenerator()
for _ in range(3):
    print(click_generator.generate())

customer_generator = CustomerGenerator()
for _ in range(3):
    print(customer_generator.generate())


{'key': None, 'value': {'customer_id': 42, 'view_time': 51, 'ts': 1786452111101}}
{'key': None, 'value': {'customer_id': 30, 'view_time': 12, 'ts': 1786452111201}}
{'key': None, 'value': {'customer_id': 53, 'view_time': 117, 'ts': 1786452111301}}
{'key': '29', 'value': {'id': 29, 'name': 'Kenneth Maxwell'}}
{'key': '26', 'value': {'id': 26, 'name': 'Cheryl Palmer'}}
{'key': '67', 'value': {'id': 67, 'name': 'Terry Nelson'}}


In [3]:
b_tn.reset()

c.recreate(click_source_str)
c.recreate(customer_source_str)
c.recreate(sink_str)

def step_fun(b_tn, source_str_offsets_dict_dict):
    size_int = b_tn.size() / 1024
    sys.stdout.write(f"\rOffsets: {source_str_offsets_dict_dict}, State size: {size_int:.2f} KB")
    sys.stdout.flush()

stop = Streams.start_streams(b_tn, step_fun=step_fun)

# stop = Streams.start_streams(built_tn, step_fun=lambda btn: print(btn.size() / 1024))

print("Streams started...")


Streams started...


(['customers'], 'streams_1786452114426')
(['clicks'], 'streams_1786452114426')


In [4]:
click_gen = ClickGenerator()
customer_gen = CustomerGenerator()

click_pr = c.producer(click_source_str)
customer_pr = c.producer(customer_source_str)

for i in range(10000):
    click_m_dict_list = [click_gen.generate() for _ in range(0, 100)]
    click_pr.produce_list(click_m_dict_list)
    #
    customer_m_dict_list = [customer_gen.generate() for _ in range(0, 100)]
    customer_pr.produce_list(customer_m_dict_list)

click_pr.close()
customer_pr.close()


Offsets: {'customers': {0: 16000}, 'clicks': {0: 15000}}, State size: 627.25 KB

'customers'

Offsets: {'customers': {0: 167000}, 'clicks': {0: 166000}}, State size: 758.14 KB

In [30]:
stop()
Streams.threads()

INFO:kafi.streams.streams:Safely stopping Streams...


Offsets: {'clicks': {0: 10000}, 'customers': {0: 10000}}, State size: 292.74 KB

INFO:kafi.streams.streams:...done.


[]

## Building Your First Kafi Streams Topology

For simplicity, we use Kafi's emulated Kafka on your local disk (replace the `root.dir` with your preferred directory where you'd like Kafi's emulated Kafka to put its files).

The interesting part is the topology specification `tn=...`. It consists of five steps (excluding the `peek` calls for debugging, you can ignore them for the time being):

1. We specify the source. Unlike Kafka Streams, each source can be on any Kafka cluster (here: `c`). The source topic name is `orders` (=`source_str`)
2. We are only interested in the `value` of the messages - hence, we just select that from the Kafka messages. In principle, you can access any field from the Kafka messages, not just the key and value but also the partition, offset, timestamp and the headers.
3. We group the messages by their `customer_id` and aggregate the respective orders and products. The "projection" of the aggregation is a record including the `customer_id`, `orders` and `product_ids`:
    * orders are just counted in `orders`
    * the product IDs are accumulated in the list `product_ids`
4. We recreate the Kafka message layout - putting the `customer_id` into the key and the aggregated record into the value.
5. We specify the sink. Again, unlike Kafka Streams, each sink can be on any Kafka cluster (here: `c` again). The sink topic name is `orders_aggregated` (=`sink_str`)

That's it. Ready to rumble.

Let's first create a generator for randomly generating orders:

In [ ]:
import random

class OrderGenerator:
    def __init__(self):
        self.order_id_int = 0
        self.customer_id_int = 0
        #
        self.ts_int = 0
        self.ts_step_int = 1

    def generate(self):
        message_dict = {
            "key": self.order_id_int,
            "value": {"id": self.order_id_int,
                      "product_id": random.randint(0, 100 - 1),
                      "customer_id": random.randint(0, 10 - 1),
                      "ts": self.ts_int},
        }
        #
        self.order_id_int += 1
        #
        self.ts_int += self.ts_step_int
        #
        return message_dict

#

gen = OrderGenerator()
for _ in range(3):
    print(gen.generate())


And then, in the next step:
1. "Build" the topology specified above to make it ready for running.
2. (Re-)create the source and sink topics.
3. Start Streams as a Python task.

...and:

4. Get the generator.
5. Set up the producer.
6. 10 times: Generate a message and produce it to Kafka.
7. Close the producer.

Since we added `peek` calls after each step, you'll see the outputs of the individual steps now as Streams picks up the produced messages.

In [ ]:
# 1. Build the topology.
b_tn = Streams.build(tn)
# 2. (Re-)create the source and sink topics.
c.recreate(source_str)
c.recreate(sink_str)
# 3. Start Streams on the built topology as a Python task.
stop = Streams.start_streams(b_tn)

#

# 4. Get the generator.
gen = OrderGenerator()
# 5. Set up the producer.
pr = c.producer(source_str)
# 6. Loop (10 times): Generate a message + produce it to Kafka.
for _ in range(10):
    message_dict = gen.generate()
    pr.produce(message_dict["value"], key=message_dict["key"])
# 7. Close the producer.
pr.close()

#

print("Streams started...")



We can now stop our Streams task...


In [ ]:
stop()

...and then print out the sink topic:

In [ ]:
c.cat(sink_str)

That's it. Congratulations - that was your first Streams topology in action :-)

That's how stream processing looks like 2026. Natural.